# Issue #05 [Data/Quality]: Diagnóstico y Tratamiento de las 6 Dimensiones de Calidad en YEB

**Subentorno:** `analytics/` | **Rama:** `feat/JMCM-analytics-nosql`

Este notebook documenta el perfilamiento de las **6 dimensiones de calidad de datos** sobre el dataset sintético con errores inducidos (`data/raw/yeb_postulaciones_calidad_raw.csv`), justificando cada decisión de tratamiento sin falsificar ni eliminar registros conflictivos a ciegas (`CA-ISS-05-01`, `CA-ISS-05-02`, `CA-ISS-05-03`).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
if (cwd / 'data' / 'raw').exists():
    ANALYTICS_ROOT = cwd
elif (cwd / 'analytics' / 'data' / 'raw').exists():
    ANALYTICS_ROOT = cwd / 'analytics'
else:
    ANALYTICS_ROOT = cwd.parent

RAW_CSV = ANALYTICS_ROOT / 'data' / 'raw' / 'yeb_postulaciones_calidad_raw.csv'
PREPARED_CSV = ANALYTICS_ROOT / 'data' / 'processed' / 'yeb_postulaciones_preparado.csv'
INCIDENCIAS_CSV = ANALYTICS_ROOT / 'outputs' / 'incidencias_calidad.csv'
CUARENTENA_CSV = ANALYTICS_ROOT / 'outputs' / 'cuarentena_identificadores_conflicto.csv'

print('Raíz analítica resuelta :', ANALYTICS_ROOT.name)
for p in (RAW_CSV, PREPARED_CSV, INCIDENCIAS_CSV, CUARENTENA_CSV):
    assert p.exists(), f'Archivo faltante: {p}'
    print('  [OK]', p.relative_to(ANALYTICS_ROOT))

Raíz analítica resuelta : analytics
  [OK] data\raw\yeb_postulaciones_calidad_raw.csv
  [OK] data\processed\yeb_postulaciones_preparado.csv
  [OK] outputs\incidencias_calidad.csv
  [OK] outputs\cuarentena_identificadores_conflicto.csv


## 1. Inspección del Dataset Crudo con Errores Inducidos (`CA-ISS-05-01`)

Se evalúan las 6 dimensiones sobre la fuente `data/raw/yeb_postulaciones_calidad_raw.csv` (`115` filas):
1. **Completitud:** Campos obligatorios de contacto (`email_contacto`, `telefono_contacto`).
2. **Unicidad:** Duplicados exactos vs. colisiones en `application_id` y `(documento_hash, rol)`.
3. **Validez:** Conversión segura con `pd.to_numeric(..., errors='coerce')`, salarios no negativos y edades en `[16, 24]` años.
4. **Consistencia:** Modalidades canónicas (`'Remoto'`, `'Híbrido'`, `'Presencial'`), ortografía de ciudades y bandera `is_protected_minor`.
5. **Oportunidad:** Ventana temporal coherente (`job_published_at <= applied_at <= updated_at <= 2026-12-31`).
6. **Trazabilidad:** Presencia de `source_system` y `audit_batch_id`.

In [2]:
df_raw = pd.read_csv(RAW_CSV, dtype=str, keep_default_na=False)
df_incidencias = pd.read_csv(INCIDENCIAS_CSV)
df_cuarentena = pd.read_csv(CUARENTENA_CSV)
df_prepared = pd.read_csv(PREPARED_CSV)

print(f'Filas crudas evaluadas          : {len(df_raw)}')
print(f'Incidencias registradas         : {len(df_incidencias)}')
print(f'Registros en cuarentena (IDs)   : {len(df_cuarentena)}')
print(f'Filas limpias en data/processed : {len(df_prepared)}')

resumen_dimensiones = (
    df_incidencias.groupby(['dimension_calidad', 'regla_evaluada', 'accion_tratamiento'])
    .size()
    .reset_index(name='casos_detectados')
)
display(resumen_dimensiones)

Filas crudas evaluadas          : 115
Incidencias registradas         : 27
Registros en cuarentena (IDs)   : 4
Filas limpias en data/processed : 93


,dimension_calidad,regla_evaluada,accion_tratamiento,casos_detectados
0,1. Completitud,COM-01: Campos obligatorios de contacto del ca...,EXCLUSION_POR_INCOMPLETITUD_CRITICA,3
1,2. Unicidad,UNI-01: Registro duplicado exacto en todas las...,DESCARTE_DUPLICADO_EXACTO,3
2,2. Unicidad,UNI-02: Colisión de clave primaria application...,AISLAMIENTO_EN_CUARENTENA,2
3,2. Unicidad,UNI-03: Cédula/documento repetido para el mism...,AISLAMIENTO_EN_CUARENTENA,2
4,3. Validez,VAL-01: Edad no numérica detectada por pd.to_n...,EXCLUSION_VALOR_NO_NUMERICO,1
5,3. Validez,VAL-02: Edad fuera del rango legal de empleabi...,EXCLUSION_FUERA_DE_RANGO_EDAD,2
6,3. Validez,VAL-03: Salario no numérico detectado por pd.t...,EXCLUSION_SALARIO_CORRUPTO,1
7,3. Validez,VAL-04: Salario negativo o rango salarial inve...,EXCLUSION_SALARIO_NEGATIVO,2
8,4. Consistencia,CON-01: Modalidad fuera del catálogo canónico ...,ESTANDARIZACION_CATALOGO -> Híbrido,1
9,4. Consistencia,CON-01: Modalidad fuera del catálogo canónico ...,ESTANDARIZACION_CATALOGO -> Presencial,1


## 2. Tratamiento de Duplicados Exactos vs. Identificadores en Conflicto (`CA-ISS-05-02` / `TEC-011`)

* **Duplicados exactos (`keep='first'`):** Filas 100% idénticas en todas las columnas producto de reintentos de extracción se descartan conservando la primera aparición.
* **Identificadores en conflicto (Cuarentena):** Cuando dos registros comparten `application_id` o `(documento_hash, rol)` pero difieren en candidato, salario o edad, **no se eliminan a ciegas ni se elige uno arbitrariamente**. Se aíslan ambos registros del par conflictivo en `outputs/cuarentena_identificadores_conflicto.csv` y se auditan en `outputs/incidencias_calidad.csv`.

In [3]:
print('=== REGISTROS AISLADOS EN CUARENTENA POR CONFLICTO DE IDENTIFICADOR ===')
display(df_cuarentena[['raw_record_id', 'application_id', 'candidate_hash', 'documento_hash', 'rol', 'candidate_age', 'ciudad_candidato', 'expected_salary_cop', 'status']])

=== REGISTROS AISLADOS EN CUARENTENA POR CONFLICTO DE IDENTIFICADOR ===


,raw_record_id,application_id,candidate_hash,documento_hash,rol,candidate_age,ciudad_candidato,expected_salary_cop,status
0,RAW-0010,APP-2026-0010,CAND-HASH-0010,DOC-CC-01010,candidate,20,Bucaramanga,2211346,submitted
1,RAW-0025,APP-2026-0025,CAND-HASH-0025,DOC-CC-01025,candidate,20,Bogotá,2697802,submitted
2,RAW-9001,APP-2026-0010,CAND-HASH-9901,DOC-CC-99901,candidate,20,Bucaramanga,3800000,accepted
3,RAW-9002,APP-2026-9002,CAND-HASH-9902,DOC-CC-01025,candidate,24,Cali,2697802,submitted


## 3. Verificación de Conversión Segura (`pd.to_numeric`) e Invariantes en `yeb_postulaciones_preparado.csv` (`CA-ISS-05-03` / `CP-045B`)

In [4]:
# Demostración de pd.to_numeric(errors='coerce') sobre la columna cruda
raw_ages_coerced = pd.to_numeric(df_raw['candidate_age'], errors='coerce')
raw_salaries_coerced = pd.to_numeric(df_raw['expected_salary_cop'], errors='coerce')
print('Valores no numéricos convertidos a NaN en candidate_age       :', int(raw_ages_coerced.isna().sum()))
print('Valores no numéricos convertidos a NaN en expected_salary_cop :', int(raw_salaries_coerced.isna().sum()))

# Verificación estricta de invariantes sobre el dataset preparado
assert df_prepared['candidate_age'].between(16, 24).all()
assert (df_prepared['expected_salary_cop'] >= 0).all()
assert (df_prepared['salary_min_offered'] >= 0).all()
assert set(df_prepared['modality'].unique()) == {'Presencial', 'Híbrido', 'Remoto'}
assert not df_prepared.duplicated(subset=['application_id']).any()
assert not df_prepared.duplicated(subset=['documento_hash', 'rol']).any()

print('\n=== RESUMEN ESTADÍSTICO DEL DATASET PREPARADO (93 FILAS SANEADAS) ===')
display(df_prepared[['candidate_age', 'salary_min_offered', 'salary_max_offered', 'expected_salary_cop', 'profile_completeness_pct']].describe().round(2))

Valores no numéricos convertidos a NaN en candidate_age       : 1
Valores no numéricos convertidos a NaN en expected_salary_cop : 1

=== RESUMEN ESTADÍSTICO DEL DATASET PREPARADO (93 FILAS SANEADAS) ===


,candidate_age,salary_min_offered,salary_max_offered,expected_salary_cop,profile_completeness_pct
count,93.00,93.00,93.00,93.00,93.00
mean,19.74,1804838.71,2296236.56,2000924.37,85.59
std,2.44,356085.64,391571.29,386603.74,7.90
min,16.00,1300000.00,1600000.00,1305884.00,72.00
25%,18.00,1450000.00,2000000.00,1658082.00,80.00
50%,19.00,1800000.00,2300000.00,1972608.00,86.00
75%,22.00,2000000.00,2600000.00,2295616.00,92.00
max,24.00,2300000.00,3000000.00,2731948.00,99.00
